In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, accuracy_score
import os
from tensorflow.keras import layers, applications

2026-01-04 04:13:13.393221: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767499993.588655    9429 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767499993.644992    9429 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767499994.114459    9429 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767499994.114504    9429 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767499994.114507    9429 computation_placer.cc:177] computation placer alr

In [2]:
MODEL1_DIR = '/kaggle/input/densenet121-ensemble/DenseNet121_Ensemble'
preprocess_fn1 = applications.densenet.preprocess_input

MODEL2_DIR = '/kaggle/input/resnet50-ensemble/ResNet50_Ensemble'
preprocess_fn2 = applications.resnet.preprocess_input

MODEL3_DIR = '/kaggle/input/vgg19-ensemble/VGG19_Ensemble'
preprocess_fn3 = applications.vgg19.preprocess_input

MODEL4_DIR = '/kaggle/input/xception-ensemble/Xception_Ensemble'
preprocess_fn4 = applications.xception.preprocess_input

MODEL5_DIR = '/kaggle/input/mobilenet-ensemble/MobileNet_Ensemble'
preprocess_fn5 = applications.mobilenet.preprocess_input

In [3]:
TRAIN_DIR = '/kaggle/input/5-fold-brain-tumor-contrast-enhanced/kfold_dataset'

In [4]:
def build_data_augmentation(SEED=24520152):
    """Create a simple data augmentation pipeline"""
    return tf.keras.Sequential([
        layers.RandomFlip('horizontal', seed=SEED),
        layers.RandomRotation(0.1, seed=SEED),
        layers.RandomZoom(0.1, seed=SEED),
        layers.RandomContrast(0.1, seed=SEED),
        layers.RandomBrightness(0.1, seed=SEED),
    ], name='data_augmentation')

In [ ]:
def get_pred_for_ensembling_model(fold_k, val_ds):
    # model1 = tf.keras.models.load_model(os.path.join(MODEL1_DIR, f'DenseNet121_block_2_fold_{fold_k}.keras'), compile=False)
    model2 = tf.keras.models.load_model(os.path.join(MODEL2_DIR, f'ResNet50_block_1_fold_{fold_k}.keras'), compile=False)
    model3 = tf.keras.models.load_model(os.path.join(MODEL3_DIR, f'VGG19_block_1_fold_{fold_k}.keras'), compile=False)
    # model4 = tf.keras.models.load_model(os.path.join(MODEL4_DIR, f'Xception_block_2_fold_{fold_k}.keras'), compile=False)
    model5 = tf.keras.models.load_model(os.path.join(MODEL5_DIR, f'MobileNet_block_1_fold_{fold_k}.keras'), compile=False)

    AUTOTUNE = tf.data.AUTOTUNE
    # val_ds1 = val_ds.map(lambda image, label: (preprocess_fn1(image), label)).prefetch(buffer_size=AUTOTUNE)
    val_ds2 = val_ds.map(lambda image, label: (preprocess_fn2(image), label)).prefetch(buffer_size=AUTOTUNE)
    val_ds3 = val_ds.map(lambda image, label: (preprocess_fn3(image), label)).prefetch(buffer_size=AUTOTUNE)
    # val_ds4 = val_ds.map(lambda image, label: (preprocess_fn4(image), label)).prefetch(buffer_size=AUTOTUNE)
    val_ds5 = val_ds.map(lambda image, label: (preprocess_fn5(image), label)).prefetch(buffer_size=AUTOTUNE)

    y_true = np.concatenate([y.numpy() for _, y in val_ds], axis=0)
    # pred1 = model1.predict(val_ds1)
    pred2 = model2.predict(val_ds2)
    pred3 = model3.predict(val_ds3)
    # pred4 = model4.predict(val_ds4)
    pred5 = model5.predict(val_ds5)
    ensemble_pred = (pred2 + pred3 + pred5) / 3.0

    return ensemble_pred

In [6]:
DATASET_CACHE = {}
def get_validation_fold(k, train_dir=TRAIN_DIR, IMG_SIZE=(224, 224), BATCH_SIZE=32, SEED=24520152):
    val_dir = os.path.join(train_dir, f'Subset_{k}')
    
    val_ds = tf.keras.utils.image_dataset_from_directory(
        val_dir,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False,
        seed=SEED,
    )
    
    AUTOTUNE = tf.data.AUTOTUNE

    return val_ds

In [7]:
def get_predictions_for_fold(k):
    valid_data = get_validation_fold(k)

    y_true = []
    for _, y in valid_data:
        y_true.extend(y.numpy())
    y_true = np.array(y_true)
    y_pred = []

    y_pred.append(get_pred_for_ensembling_model(k, valid_data))
    
    y_pred = np.array(y_pred)
    return y_true, y_pred

In [8]:
def macro_specificity(y_true, y_pred, num_classes):
    cm = confusion_matrix(y_true, y_pred)
    spec = []

    for i in range(num_classes):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = cm.sum() - (TP + FN + FP)

        spec.append(TN / (TN + FP + 1e-8))

    return np.array(spec)

In [9]:
def evaluate_fold(k):
    y_true, y_pred = get_predictions_for_fold(k)

    model_results = []
    for i in range(len(y_pred)):
        y_pred_label = np.argmax(y_pred[i], axis=1)
        acc = accuracy_score(y_true, y_pred_label)
        precision = precision_score(y_true, y_pred_label, average='macro', zero_division=0)
        recall = recall_score(y_true, y_pred_label, average='macro', zero_division=0)
        f1 = f1_score(y_true, y_pred_label, average='macro', zero_division=0)
        spec_per_class = macro_specificity(y_true, y_pred_label, 3)
        specificity = np.mean(spec_per_class)

        model_results.append({
            'Accuracy': np.array(acc),
            'Precision': np.array(precision),
            'Recall': np.array(recall),
            'F1-Score': np.array(f1),
            'Specificity': np.array(specificity)
        })
    return np.array(model_results)

In [10]:
five_fold_results = []
for i in range(1, 6):
    five_fold_results.append(evaluate_fold(i))

five_fold_results = np.array(five_fold_results)

num_fold = 5
num_model = 1
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Specificity']

avg_metrics = {}

for m in range(num_model):
    avg_metrics[m] = {}
    for metric in metric_names:
        values = np.array([
            five_fold_results[k][m][metric] for k in range(num_fold)
        ])
        avg_metrics[m][metric] = values.mean(axis=0)*100

Found 542 files belonging to 3 classes.


I0000 00:00:1767500001.736810    9429 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
I0000 00:00:1767500013.281054    9471 service.cc:152] XLA service 0x7b156c001f40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1767500013.281088    9471 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1767500014.128108    9471 cuda_dnn.cc:529] Loaded cuDNN version 91002


 4/17 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step

I0000 00:00:1767500017.483395    9471 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


17/17 ━━━━━━━━━━━━━━━━━━━━ 12s 363ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 15s 480ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 11s 318ms/step
Found 679 files belonging to 3 classes.
22/22 ━━━━━━━━━━━━━━━━━━━━ 10s 270ms/step
22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 200ms/step
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 234ms/step
Found 572 files belonging to 3 classes.
18/18 ━━━━━━━━━━━━━━━━━━━━ 10s 347ms/step
18/18 ━━━━━━━━━━━━━━━━━━━━ 9s 472ms/step
18/18 ━━━━━━━━━━━━━━━━━━━━ 8s 309ms/step
Found 628 files belonging to 3 classes.
20/20 ━━━━━━━━━━━━━━━━━━━━ 10s 300ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 8s 354ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 8s 289ms/step
Found 643 files belonging to 3 classes.
21/21 ━━━━━━━━━━━━━━━━━━━━ 10s 273ms/step
21/21 ━━━━━━━━━━━━━━━━━━━━ 4s 166ms/step
21/21 ━━━━━━━━━━━━━━━━━━━━ 7s 229ms/step


In [ ]:
row_names = ['ensemble-vgg19-resnet50-mobilenet']
column_names = ['Recall', 'Specificity', 'Precision', 'F1-Score', 'Accuracy']

df_result = pd.DataFrame(avg_metrics).T
df_result.index = row_names
df_result = df_result[column_names]
df_result = df_result.round(2)
df_result

,Recall,Specificity,Precision,F1-Score,Accuracy
ensemble-vgg19-resnet50-mobilenet,92.85,96.92,93.32,92.98,93.9


In [12]:
latex_table = df_result.to_latex(
    multicolumn=True,
    multirow=True,
    float_format="%.2f",
    label="tab:sens_spec"
)

print(latex_table)

\begin{table}
\label{tab:sens_spec}
\begin{tabular}{lrrrrr}
\toprule
 & Recall & Specificity & Precision & F1-Score & Accuracy \\
\midrule
ensemble-vgg19-resnet50-mobilenet & 92.85 & 96.92 & 93.32 & 92.98 & 93.90 \\
\bottomrule
\end{tabular}
\end{table}

